# DeFi-LSA D2 Real Data Collection on Kaggle

This notebook clones a GitHub repository, loads credentials from Kaggle Secrets, installs the package, writes runtime-only source configuration, runs tests, performs connectivity checks, and launches D2 collection.

## Required Kaggle Secrets
- `GITHUB_REPO_URL`: HTTPS URL of the public repository, or a URL that does not embed credentials.
- `GITHUB_TOKEN`: optional, only for a private repository.
- `ETH_RPC_URL`: Ethereum mainnet JSON-RPC endpoint.
- `ETHERSCAN_API_KEY`: Etherscan API key.
- `COINGECKO_API_KEY`: optional CoinGecko Demo key. Keyless mode may be used at lower rate limits.

Enable Internet in the Kaggle notebook settings before running.


In [ ]:
from pathlib import Path
import os, json, subprocess, sys
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
def get_secret(name, required=False):
    try:
        value = secrets.get_secret(name)
    except Exception:
        value = None
    if required and not value:
        raise RuntimeError(f"Missing required Kaggle Secret: {name}")
    return value

repo_url = 'https://github.com/XQuyTran/DeFi-LSA.git'
repo_branch = 'feature/e7-e8-runtime'
github_token = get_secret("GITHUB_TOKEN")
os.environ["ETH_RPC_URL"] = get_secret("ETH_RPC_URL", required=True)
os.environ["ETHERSCAN_API_KEY"] = get_secret("ETHERSCAN_API_KEY", required=True)
cg_key = get_secret("COINGECKO_API_KEY")
if cg_key:
    os.environ["COINGECKO_API_KEY"] = cg_key

WORK = Path("/kaggle/working")
REPO = WORK / "defi-lsa"
OUTPUT = WORK / "defi-lsa-output"
restored_candidates = [
    WORK / "defi-lsa-output",
    WORK / "input" / "defi-lsa-output",
]
for candidate in restored_candidates:
    if any((candidate / marker).exists() for marker in ("data", "checkpoints", "reports")):
        OUTPUT = candidate
        break
OUTPUT.mkdir(parents=True, exist_ok=True)
print(f"Runtime root selected: {OUTPUT}")
print("Secrets loaded; values are not displayed.")

In [ ]:
import urllib.request
urllib.request.urlopen("https://github.com", timeout=15).read(64)
print("Internet connectivity: OK")


In [ ]:
import subprocess, os, base64
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", 
                    "-c", "http.extraheader=Authorization: Bearer " + base64.b64encode(github_token),
                    "clone", "--depth", "1", '--branch', repo_branch, repo_url, str(REPO)], check=True, env=os.environ)
print(subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[dev,parquet,aws]"], check=True)
print("Package installation: OK")


In [ ]:
runtime_config = {
  "etherscan_api_url": "https://api.etherscan.io/v2/api",
  "coingecko_api_url": "https://api.coingecko.com/api/v3",
  "gdelt_enabled": True,
  "request_timeout_seconds": 30,
  "maximum_attempts": 5,
  "requests_per_second": 2.0,
  "initial_log_chunk_size": 2000,
  "minimum_log_chunk_size": 10,
  "maximum_log_chunk_size": 20000
}
config_path = OUTPUT / "runtime_sources.json"
config_path.write_text(json.dumps(runtime_config, indent=2), encoding="utf-8")
print(f"Secret-free runtime configuration written to {config_path}")

In [ ]:
from defi_lsa.runtime import preflight_internet, preflight_output_root

preflight = preflight_output_root(OUTPUT, minimum_free_bytes=1_000_000_000)
if not preflight["passed"]:
    raise RuntimeError(f"Runtime preflight failed: {preflight}")
internet = preflight_internet("https://github.com")
if not internet["reachable"]:
    raise RuntimeError(f"Internet preflight failed: {internet}")
print({"output_preflight": preflight, "internet_preflight": internet})

In [ ]:
# This command must perform real probes after Epic E2 is implemented.
subprocess.run([
    sys.executable, "-m", "defi_lsa", "connectivity",
    "--event", "euler",
    "--config", str(config_path),
    "--source", "rpc"
], cwd=REPO, check=True)


## Smoke collection
Run a small real-data collection before a full event window. The CLI must be wired by Epic E3. If the current repository only writes a summary, stop and implement E3 rather than interpreting the command as a successful collection.


In [ ]:
SMOKE_RANGES = {
    "euler": {
        "one_block": (16818024, 16818024),
        "100_blocks": (16817925, 16818024),
    },
    "kyber": {
        "one_block": (18630392, 18630392),
        "100_blocks": (18630293, 18630392),
    },
}
SMOKE_AUDITS = {}
for event, ranges in SMOKE_RANGES.items():
    for label, (start_block, end_block) in ranges.items():
        smoke_output = OUTPUT / "smoke" / f"{event}_{label}"
        subprocess.run([
            sys.executable, "-m", "defi_lsa", "collect",
            "--event", event,
            "--stage", "all",
            "--source", "auto",
            "--start-block", str(start_block),
            "--end-block", str(end_block),
            "--config", str(config_path),
            "--output", str(smoke_output),
            "--resume"
        ], cwd=REPO, check=True)
        subprocess.run([
            sys.executable, "-m", "defi_lsa", "audit",
            "--event", event,
            "--stage", "d2",
            "--output", str(smoke_output),
            "--strict"
        ], cwd=REPO, check=True)
        SMOKE_AUDITS[event, label] = smoke_output / event / "d2_collection_audit.json"

In [ ]:
subprocess.run([
    sys.executable, "-m", "defi_lsa", "audit",
    "--event", "euler",
    "--stage", "d2",
    "--output", str(OUTPUT)
], cwd=REPO, check=True)


## Full event runs
Uncomment only after the smoke audit reports `GO` or an approved `CONDITIONAL_GO`, and after Euler/Curve/Kyber registries contain complete windows, incident legs, token mappings, and public-source queries.


In [ ]:
EVENTS = ["euler", "curve", "kyber"]
RUN_FULL = False
if RUN_FULL:
    for event in EVENTS:
        smoke_audits = [
            path for (audit_event, _label), path in SMOKE_AUDITS.items()
            if audit_event == event
        ]
        if not smoke_audits:
            raise RuntimeError(f"Missing smoke audits for {event}")
        for audit_path in smoke_audits:
            audit = json.loads(audit_path.read_text(encoding="utf-8"))
            if audit.get("status") != "GO":
                raise RuntimeError(f"Full run blocked for {event}: {audit.get('status')}")
        subprocess.run([
            sys.executable, "-m", "defi_lsa", "collect",
            "--event", event,
            "--stage", "all",
            "--source", "auto",
            "--config", str(config_path),
            "--output", str(OUTPUT),
            "--resume"
        ], cwd=REPO, check=True)
        subprocess.run([
            sys.executable, "-m", "defi_lsa", "audit",
            "--event", event,
            "--stage", "d2",
            "--output", str(OUTPUT),
            "--strict"
        ], cwd=REPO, check=True)

In [ ]:
# Sanity-check generated artifacts without exposing secrets.
for path in sorted(OUTPUT.rglob("*")):
    if path.is_file() and path != config_path:
        print(path.relative_to(OUTPUT), path.stat().st_size)
